# Homing quality review for the new detector

This notebook loads a session, runs the new automatic homing detector, compares it against manual BORIS labels when available, and lets you inspect each detected homing with a `syd` viewer.

The main views are:
- XY trajectory
- Speed
- Head direction
- Automatic versus manual label overlap

In [77]:
%load_ext autoreload
%autoreload 2
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from syd import make_viewer

from JR_test_scripts.homing_logic.homings_new_logic import get_Homings
from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from settings.settings_analyze_behave import settings_analyze_behave as settings
from behave_analysis.utils.arena_plotting import Arena

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_shelt_17aug, JAL3_mush_21aug, JAL3_flip1_22aug, JAL3_flip2_25aug, JAL3_flip3_29aug, JAL3_flip4_1sept, JAL3_flip5_4sept, JAL3_flip6_7sept

from behave_analysis.database.Experiments.JAL004_ex import JAL4_shelt_17aug, JAL4_mush_18aug, JAL4_flip1_21Aug, JAL4_mush2_22Aug, JAL4_flip3_28aug, JAL4_flip4_3Sept, JAL4_flip5_11Sept, JAL4_flip6_19Sept

from behave_analysis.database.Experiments.JAL005_ex import JAL5_shelt_2Sept, JAL5_barr_5Sept, JAL5_flip1_8Sept, JAL5_flip3_21Sept, JAL5_mush_3oct

from behave_analysis.database.Experiments.JAL006_ex import JAL6_hab_1mar, JAL6_shelt_4mar, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip6_28mar, JAL6_flip7_1apr, JAL6_flip8_5apr


from behave_analysis.database.Experiments.JAL007_ex import JAL7_hab_1mar, JAL7_empty_shelter_5mar, JAL7_flip2_12mar, JAL7_flip3_15mar, JAL7_flip4_19mar, JAL7_flip5_22mar, JAL7_flip7_4apr, JAL7_flip8_9apr, JAL7_flip9_16apr, JAL7_flip10_23apr, JAL7_tiny_30apr

from behave_analysis.database.Experiments.JAL008_ex import (
    JAL8_shelt_22apr,
    JAL8_flip1_25apr,
    JAL8_flip2_29apr,
    JAL8_tiny_3may,
    JAL8_flip3_7may,
    JAL8_flip4_10may,
    JAL8_flip5_14may,
    JAL8_tiny2_21may,
)

experiments_objects = {"JAL4_3rdSept": JAL4_flip4_3Sept,
        "JAL4_19thSept": JAL4_flip6_19Sept,
        "JAL4_28aug": JAL4_flip3_28aug,
        "JAL4_11thSept": JAL4_flip5_11Sept,
        "JAL5_8thSept": JAL5_flip1_8Sept,
        "JAL5_21stSept": JAL5_flip3_21Sept,
        "JAL6_28mar": JAL6_flip6_28mar, 
        "JAL6_flip4_21mar": JAL6_flip4_21mar, 
        "JAL6_flip3_18mar": JAL6_flip3_18mar, 
        "JAL6_flip5_25mar": JAL6_flip5_25mar, 
        "JAL7_sesh8_9apr": JAL7_flip8_9apr, # almost no homings
        "JAL7_flip5_22mar": JAL7_flip5_22mar, 
        "JAL7_flip2_12mar": JAL7_flip2_12mar, 
        "JAL7_sesh9_16apr": JAL7_flip9_16apr, 
        "JAL7_23apr": JAL7_flip10_23apr,
        "JAL8_flip1_25apr": JAL8_flip1_25apr, 
        "JAL8_flip2_29apr": JAL8_flip2_29apr, 
        "JAL8_flip3_7may": JAL8_flip3_7may, 
        "JAL8_14may": JAL8_flip5_14may, 
        "JAL8_flip4_10may": JAL8_flip4_10may}

e = 19

cond_list = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
session_name = list(experiments_objects.keys())[e]
exp = experiments_objects[session_name]
session = Process(exp).load_session()

2026-06-10 20:47:36.931 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now


In [80]:
tracking_data = open_tracking_data(session)
video_df_path = os.path.join(session.base_path, session.processed_path, 'full_video_dataframe.csv')
video_df = pl.read_csv(video_df_path)

detector = get_Homings(settings, session, video_df=video_df)
n_frames = len(tracking_data['avg_loc'])

2026-06-10 20:48:29.776 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings_new_logic:get_homings:70 - Extracting homings automatically with new logic...
2026-06-10 20:48:29.816 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings_new_logic:get_homing_speed:174 - Homing speed is too high, check tracking data


AttributeError: 'Settings_analyze_behave' object has no attribute 'homing_min_frames_between_trials'

In [73]:
def load_manual_mask(session, n_frames):
    boris_path = os.path.join(session.base_path, session.processed_path, 'Borris', 'scored_homings.csv')
    manual_mask = np.zeros(n_frames, dtype=bool)
    if not os.path.isfile(boris_path):
        return manual_mask, pd.DataFrame()

    df = pd.read_csv(boris_path)
    starts = df.loc[df['Behavior type'] == 'START', 'Image index'].to_numpy(dtype=int) - 1
    stops = df.loc[df['Behavior type'] == 'STOP', 'Image index'].to_numpy(dtype=int) - 1
    for onset, offset in zip(starts, stops):
        onset = max(int(onset), 0)
        offset = min(int(offset), n_frames - 1)
        manual_mask[onset:offset + 1] = True
    return manual_mask, df

def build_auto_mask(onsets, offsets, n_frames):
    mask = np.zeros(n_frames, dtype=bool)
    for onset, offset in zip(onsets, offsets):
        onset = max(int(onset), 0)
        offset = min(int(offset), n_frames - 1)
        mask[onset:offset + 1] = True
    return mask

manual_mask, manual_df = load_manual_mask(session, n_frames)
auto_mask = build_auto_mask(auto_homings.onset_frames, auto_homings.offset_frames, n_frames)

tp = int(np.sum(auto_mask & manual_mask))
fp = int(np.sum(auto_mask & ~manual_mask))
fn = int(np.sum(~auto_mask & manual_mask))
precision = tp / (tp + fp) if (tp + fp) else np.nan
recall = tp / (tp + fn) if (tp + fn) else np.nan

print(f'Automatic homings: {len(auto_homings.onset_frames)}')
print(f'Manual frames: {manual_mask.sum()}')
print(f'Auto frames: {auto_mask.sum()}')
print(f'Frame precision: {precision:.3f}')
print(f'Frame recall: {recall:.3f}')

2026-06-10 20:38:15.372 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings_new_logic:get_homings:101 - Extracting homings automatically with hybrid logic
2026-06-10 20:38:18.213 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings_new_logic:find_homing_segments:342 - Accepted 13 homing segments (rejected 486)
2026-06-10 20:38:19.171 | SUCCESS  | behave_analysis.analyze.behaviour.homings_escapes.homings_new_logic:save_session:526 - Homings object pickle saved


Detector run with debug_settings


In [82]:
all_conditions

NameError: name 'all_conditions' is not defined

In [81]:
viewer = make_viewer()
max_trial = max(len(auto_homings.onset_frames) - 1, 0)
viewer.add_integer('trial', min=0, max=max_trial)

all_onsets = np.append(auto_homings.onset_frames, manual_df.loc[manual_df['Behavior type'] == 'START', 'Image index'].to_numpy(dtype=int) - 1)
all_offsets = np.append(auto_homings.offset_frames, manual_df.loc[manual_df['Behavior type'] == 'STOP', 'Image index'].to_numpy(dtype=int) - 1)
manual_condition = np.empty_like(auto_homings.condition)
for i, ons in enumerate(manual_df.loc[manual_df['Behavior type'] == 'START', 'Image index'].to_numpy(dtype=int) - 1):
    manual_condition[i] = auto_homings.condition[np.argmin(np.abs(auto_homings.onset_frames - ons))]
all_conditions = np.append(auto_homings.condition, manual_condition)
sort_idx = np.argsort(all_onsets)
all_onsets = all_onsets[sort_idx]
all_offsets = all_offsets[sort_idx]
all_conditions = all_conditions[sort_idx]

def plot(state):
    idx = int(state['trial'])
    if len(auto_homings.onset_frames) == 0:
        fig, ax = plt.subplots(1, 1, figsize=(6, 4))
        ax.text(0.5, 0.5, 'No automatic homings detected', ha='center', va='center')
        ax.set_axis_off()
        return fig

    onset = int(all_onsets[idx])
    offset = int(all_offsets[idx])
    pad = int(session.video.fps)
    start = max(0, onset - pad)
    stop = min(n_frames, offset + pad)

    frames = np.arange(start, stop)
    time = (frames - start) / session.video.fps
    x = tracking_data['avg_loc'][start:stop, 0]
    y = tracking_data['avg_loc'][start:stop, 1]
    speed = tracking_data['avg_Velocity'][start:stop]
    hdir = tracking_data['hdir'][start:stop]
    auto_win = auto_mask[start:stop]
    manual_win = manual_mask[start:stop]

    fig, axs = plt.subplots(3, 2, figsize=(10, 11), gridspec_kw={'height_ratios': [2, 1, 1]})

    axs[0, 0].plot(x, y, color='0.75', lw=1)
    axs[0, 0].scatter(x[auto_win], y[auto_win], s=8, color='tab:blue', alpha=0.8, label='auto')
    axs[0, 0].scatter(x[0], y[0], s=35, color='green', label='window start')
    axs[0, 0].scatter(x[-1], y[-1], s=35, color='red', label='window end')
    Arena(ax = axs[0, 0], condition = all_conditions[idx],
                barrier_coordinates = session.barrier_location[:-1], full_image = False)
    axs[0, 1].scatter(x[manual_win], y[manual_win], s=8, color='tab:orange', alpha=0.8, label='manual')
    Arena(ax = axs[0, 1], condition = all_conditions[idx],
                barrier_coordinates = session.barrier_location[:-1], full_image = False)
    axs[0, 0].set_title(f'Trial {idx} | onset={onset} offset={offset} | condition={all_conditions[idx]}')
    axs[0, 0].set_xlim(0, 1024)
    axs[0, 0].set_ylim(0, 1024)
    axs[0, 0].set_aspect('equal')
    axs[0, 0].invert_yaxis()
    axs[0, 0].legend(loc='upper right', fontsize=8)

    axs[1, 0].plot(time, speed, color='black', lw=1)
    axs[1, 0].fill_between(time, 0, speed, where=auto_win, color='tab:blue', alpha=0.25)
    axs[1, 1].plot(time, speed, color='black', lw=1)
    axs[1, 1].fill_between(time, 0, speed, where=manual_win, color='tab:orange', alpha=0.18)
    axs[1, 0].axvline((onset-start)/session.video.fps, color='tab:green', ls='--', lw=1)
    axs[1, 0].axvline((offset-start)/session.video.fps, color='tab:red', ls='--', lw=1)
    axs[1, 0].set_ylabel('speed')

    axs[2, 0].plot(time, hdir, color='slategray', lw=1)
    axs[2, 0].fill_between(time, np.min(hdir), np.max(hdir), where=auto_win, color='tab:blue', alpha=0.12)
    axs[2, 1].plot(time, hdir, color='slategray', lw=1)
    axs[2, 1].fill_between(time, np.min(hdir), np.max(hdir), where=manual_win, color='tab:orange', alpha=0.10)
    axs[2, 0].axvline((onset-start)/session.video.fps, color='tab:green', ls='--', lw=1)
    axs[2, 0].axvline((offset-start)/session.video.fps, color='tab:red', ls='--', lw=1)
    axs[2, 0].set_ylabel('hdir')
    axs[2, 0].set_xlabel('frame')

    overlap = int(np.sum(auto_mask[onset:offset + 1] & manual_mask[onset:offset + 1]))
    manual_count = int(np.sum(manual_mask[onset:offset + 1]))
    auto_count = int(np.sum(auto_mask[onset:offset + 1]))
    axs[0, 1].set_title(f'auto frames={auto_count} | manual frames={manual_count} | overlap={overlap}')

    plt.tight_layout()
    return fig

viewer.set_plot(plot)
viewer.show()

KeyError: 'Behavior type'